# 8 - LightGBM MLflow serving experiment

Runs in the **Jupyter container on node-serve-lightgbm**.

This notebook validates the separate LightGBM serving path that loads the training teammate's model from MLflow.


In [ ]:
import time, json, random, concurrent.futures
import numpy as np
import requests

BASE_URL = "http://fastapi_lgbm:8000"


In [ ]:
# Health check
print(requests.get(f"{BASE_URL}/health").json())


In [ ]:
def make_payload(n_songs=10):
    return {
        "session_id": f"session_{random.randint(1, 100000)}",
        "user_features": {
            "user_skip_rate": round(random.random(), 4),
            "user_favorite_genre_encoded": random.randint(0, 20),
            "user_watch_time_avg": round(random.uniform(10, 240), 2),
        },
        "candidate_songs": [
            {
                "video_id": f"video_{i}",
                "release_year": random.randint(1970, 2026),
                "context_segment": random.randint(0, 10),
                "genre_encoded": random.randint(0, 20),
                "subgenre_encoded": random.randint(0, 3000),
            }
            for i in range(n_songs)
        ],
    }


In [ ]:
# Smoke test
payload = make_payload(5)
r = requests.post(f"{BASE_URL}/queue", json=payload)
r.raise_for_status()
body = r.json()
print(json.dumps(body, indent=2))

with open("smoke_output_lightgbm.json", "w") as f:
    json.dump(body, f, indent=2)


In [ ]:
# Sequential validation - 100 requests
N = 100
latencies = []

for _ in range(N):
    t0 = time.time()
    r = requests.post(f"{BASE_URL}/rank", json=make_payload())
    latencies.append(time.time() - t0)
    assert r.status_code == 200, r.text

print(f"Sequential ({N} requests):")
print(f"  p50: {np.percentile(latencies, 50) * 1000:.2f} ms")
print(f"  p95: {np.percentile(latencies, 95) * 1000:.2f} ms")
print(f"  p99: {np.percentile(latencies, 99) * 1000:.2f} ms")
print(f"  Throughput: {N / sum(latencies):.2f} req/sec")


In [ ]:
# Concurrent validation - 1000 requests, 16 workers
N_WORKERS = 16
N_REQUESTS = 1000

def send_one(_):
    t0 = time.time()
    r = requests.post(f"{BASE_URL}/rank", json=make_payload())
    return time.time() - t0, r.status_code

t_start = time.time()
with concurrent.futures.ThreadPoolExecutor(max_workers=N_WORKERS) as pool:
    results = list(pool.map(send_one, range(N_REQUESTS)))
t_wall = time.time() - t_start

lats = [r[0] for r in results]
errors = sum(1 for r in results if r[1] != 200)

print(f"Concurrent ({N_REQUESTS} requests, {N_WORKERS} workers):")
print(f"  p50: {np.percentile(lats, 50) * 1000:.2f} ms")
print(f"  p95: {np.percentile(lats, 95) * 1000:.2f} ms")
print(f"  p99: {np.percentile(lats, 99) * 1000:.2f} ms")
print(f"  Throughput: {N_REQUESTS / t_wall:.2f} req/sec")
print(f"  Errors: {errors} ({errors / N_REQUESTS * 100:.1f}%)")


Official saved evaluation commands are run from the VM host shell:

Baseline:

```bash
cd ~/smartqueue/serving/docker
MODEL_DIR=~/smartqueue/training/artifacts MODEL_VERSION=lightgbm_v4_baseline UVICORN_WORKERS=1 \
MLFLOW_TRACKING_URI=http://129.114.24.226:30500 \
MODEL_URI=runs:/2ce32ba692c54095b4307ae8eb7ba508/model \
docker compose --profile eval -f docker-compose-lightgbm.yaml run --rm eval sh run_evaluation.sh lightgbm_v4_baseline
```

Concurrent:

```bash
cd ~/smartqueue/serving/docker
MODEL_DIR=~/smartqueue/training/artifacts MODEL_VERSION=lightgbm_v4_concurrent UVICORN_WORKERS=4 \
MLFLOW_TRACKING_URI=http://129.114.24.226:30500 \
MODEL_URI=runs:/2ce32ba692c54095b4307ae8eb7ba508/model \
docker compose --profile eval -f docker-compose-lightgbm.yaml run --rm eval sh run_evaluation.sh lightgbm_v4_concurrent
```
